# Hybrid ALNS Performance Evaluation

## I. Purpose

This notebook defines a benchmark procedure to evaluate the Hybrid ALNS solver,
which uses an offline repair model to guide neighbourhood reconstruction.

## II. Experimental Preconditions and Reproducibility Controls

Required artefacts:
- trained model `bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/models/repair_model_v2.pkl`
- benchmark instance directory (`datasets/` at the repo root)
- fixed random seed and defined iteration / time budgets

In [ ]:
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
    hybrid_alns_solver,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.models import (
    load_repair_model,
)
from bin_packing_optimization.utilities.benchmarking import Benchmark, create_benchmark
from bin_packing_optimization.utilities.graphing import create_graphs
from bin_packing_optimization.utilities.statistics import (
    load_results,
    summarize,
    summarize_by_size,
)

## III. Hybrid Run (ALNS with Offline Repair Model)

Runs the full benchmark via the `Benchmark` class.

In [ ]:
model_bundle = load_repair_model("repair_model_v2.pkl")

In [ ]:
benchmark: Benchmark = create_benchmark(
    dataset_key="falkenauer-t",
    solver_module=hybrid_alns_solver,
)
benchmark.run(
    method=None,
    method_args={
        "max_iterations": 2000,
        "model_bundle": model_bundle,
    },
)
csv_path = benchmark.save_results_to_csv()

## IV. Summary Statistics

In [ ]:
rows = load_results(csv_path)
summary = summarize(rows)
by_size = summarize_by_size(rows)

print("Overall")
for k, v in summary.items():
    print(f"  {k:<30s}: {v:.4f}" if isinstance(v, float) else f"  {k:<30s}: {v}")

print("\nBy instance size")
for s in by_size:
    print(
        f"  n={s.num_items:4d} | completed={s.completed}/{s.instances}"
        f" | avg_time={s.avg_time_s:.4f}s | avg_gap={s.avg_gap:.3f}"
    )

## V. Graphs

In [ ]:
graph_paths = create_graphs(csv_path)
for p in graph_paths:
    print(p)